<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/day07-discussion-2.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Day 7 — In-class discussion problem (2 of 3)

Discuss **as a group first**, write down a guess, then run the code to
check it before presenting.

## Not every protein looks like hemoglobin

Both hemoglobin's beta chain (80.8% helix, 0% strand) and myoglobin
(79.7% helix, 0% strand) are almost entirely alpha-helical, with zero
beta content. That might make it tempting to guess "most proteins are
mostly helix" — but that's just because globins happen to be an
all-alpha family.

Green Fluorescent Protein (GFP, PDB `1EMA`) is one of the most famous
counter-examples in structural biology: it's built from a barrel of beta
strands wrapped around a single central helix that carries the
fluorophore.

**As a group, before running anything:** write down your own guess for
GFP's percent helix and percent strand (they don't need to add to 100 —
the rest is coil). Then run the cell below.

In [1]:
import subprocess
import os
from Bio.PDB import PDBList, PDBParser

pdbl = PDBList()
gfp_path = pdbl.retrieve_pdb_file("1ema", pdir=".", file_format="pdb")

pymol_script = '''
from pymol import cmd

cmd.load("PDBPATH", "gfp")
cmd.dss("gfp")

ss_list = []
cmd.iterate("gfp and polymer and name CA",
            "ss_list.append(ss)", space={"ss_list": ss_list})

total = len(ss_list)
helix = sum(1 for s in ss_list if s == "H")
strand = sum(1 for s in ss_list if s == "S")
print("TOTAL", total)
print("HELIX", helix)
print("STRAND", strand)
print("PCT_HELIX", round(100 * helix / total, 1))
print("PCT_STRAND", round(100 * strand / total, 1))
'''.replace("PDBPATH", gfp_path)

with open("_disc2_pymol.py", "w") as f:
    f.write(pymol_script)

pymol_env = dict(os.environ)
pymol_env["PATH"] = "/usr/bin:/bin:" + pymol_env.get("PATH", "")

result = subprocess.run(["/usr/bin/pymol", "-cq", "_disc2_pymol.py"],
                         capture_output=True, text=True, env=pymol_env)
for line in result.stdout.splitlines():
    if line.startswith(("TOTAL", "HELIX", "STRAND", "PCT")):
        print(line)

os.remove("_disc2_pymol.py")

TOTAL 225
HELIX 22
STRAND 121
PCT_HELIX 9.8
PCT_STRAND 53.8


**Discussion point.** GFP comes out at **9.8% helix, 53.8% strand** —
almost the mirror image of the globins. This is exactly why "predict
secondary structure" can't be solved by any single fixed rule like
"assume mostly helix": the correct answer depends entirely on which
protein family you're looking at, which is precisely why Day 6's PHD
method needs to *learn* position-specific patterns from real evolutionary
data (profiles) rather than applying one global assumption everywhere.

**Follow-up, as a group:** proline and glycine are both known to disrupt
regular backbone geometry (proline's ring locks its own backbone angle;
glycine's small side chain makes the backbone unusually flexible) and so
rarely appear in the *middle* of alpha helices or beta strands. If you
were handed a new sequence, with no structure solved yet, that was
unusually rich in proline and glycine throughout, what would you predict
about its secondary-structure content — and why?